# M3-B1 — Fusion de deux sources + colonne de provenance

> ~30-40 min. Le geste **exact** que le cas d'usage certif attend côté
> données : réunir **deux exports comparables** (même métier, deux
> origines) dans un seul tableau, **sans perdre d'où vient chaque ligne**.

Ici : deux sites d'Acerox t'envoient chacun un export de capteurs. Même
idée, mais ce **ne sont pas** exactement les mêmes fichiers. Ton travail :
les empiler proprement et repérer ce qui cloche.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('..') / 'data'
a = pd.read_csv(DATA_DIR / 'capteurs_site_A.csv')
b = pd.read_csv(DATA_DIR / 'capteurs_site_B.csv')
print('A', a.shape, '| colonnes', list(a.columns))
print('B', b.shape, '| colonnes', list(b.columns))

A (25, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'debit_l_min']
B (22, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'firmware']


## 1. Le réflexe naïf (à ne PAS garder)

On empile les deux sans réfléchir. Regarde bien le résultat.

In [8]:
naif = pd.concat([a, b], ignore_index=True)
naif.info()
# TODO — retrouve, dans `naif`, quelles lignes viennent de A et lesquelles
# de B. Tu ne peux pas, hein ? C'est exactement le problème.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ts              47 non-null     object 
 1   machine_id      47 non-null     object 
 2   temp_c          47 non-null     float64
 3   vibration_mm_s  47 non-null     float64
 4   debit_l_min     25 non-null     float64
 5   firmware        22 non-null     object 
dtypes: float64(3), object(3)
memory usage: 2.3+ KB


> **Ce qui vient de casser** (à formuler toi-même) :
>
> - impossible de dire **d'où vient** chaque ligne ;
> - une colonne (`debit_l_min`) est pleine de `NaN` pour la moitié des
>   lignes — pourquoi ?
> - `temp_c` mélange peut-être **deux échelles** sans prévenir…

## 2. Le bon geste : marquer la provenance AVANT de concaténer

Une colonne `source` = une **fiche d'identité** de chaque ligne. C'est la
base de la traçabilité (RGPD, réconciliation, filtrage par scénario ensuite).

In [3]:
a2 = a.assign(source='site_A')
b2 = b.assign(source='site_B')
fusion = pd.concat([a2, b2], ignore_index=True)
fusion['source'].value_counts()

source
site_A    25
site_B    22
Name: count, dtype: int64

## 3. Maintenant, la provenance te fait VOIR les pièges

Fusionner n'est pas empiler. Trois choses à débusquer avec la colonne `source`.

In [9]:
# Piège A — mêmes machine_id des deux côtés : le « M04 » de A est-il
# le même équipement que le « M04 » de B ? (indice : non)
communs = set(a['machine_id']) & set(b['machine_id'])
print('machine_id présents dans les DEUX sites :', sorted(communs))
# TODO — que faire ? préfixer par la source ? (ex. 'site_A::M04')

machine_id présents dans les DEUX sites : ['M04', 'M05']


In [5]:
# Piège B — même colonne temp_c, mais l'échelle diffère selon la source.
fusion.groupby('source')['temp_c'].describe()[['mean', 'min', 'max']]
# TODO — une des deux sources est en °F (unité oubliée). Laquelle ?
# Sans la colonne `source`, cette anomalie était invisible.

,mean,min,max
source,,,
site_A,68.544000,64.2,76.6
site_B,153.681818,142.9,166.2


In [6]:
# Piège C — debit_l_min n'existe que pour un site -> NaN pour l'autre.
fusion.groupby('source')['debit_l_min'].apply(lambda s: s.isna().mean())
# TODO — colonne réellement absente, ou juste pas transmise ? -> question client.

source
site_A    0.0
site_B    1.0
Name: debit_l_min, dtype: float64

## Ce que tu dois en retenir

- **Fusionner ≠ empiler.** Deux fichiers « du même genre » cachent des
  écarts de schéma, d'unité et de clé.
- La colonne **`source` (provenance)** n'est pas cosmétique : c'est elle
  qui rend les écarts **visibles** et traçables — et qui te laissera plus
  tard filtrer par origine (scénarios, RGPD, réconciliation).
- Reporte une phrase dans `identification_sources.md` : *« Les sources X
  et Y ont été réunies avec une colonne `source` ; écarts constatés : … »*